# 项目文件

## 说明

本次项目我使用了随机森林算法，通过分析歌曲的音乐特征来猜测其是否能成为热门歌曲，代码中的注释后续会逐渐完善

## 代码

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score,confusion_matrix
from sklearn.ensemble import RandomForestClassifier


df = pd.read_csv("songs_normalize.csv")
main_genres = ["pop","rock","hip hop","Dance/Electronic"]
df["genre_main"] = df["genre"].apply(
    lambda x:x if x in main_genres else "other"
)
genre_dummies = pd.get_dummies(df["genre_main"],prefix="genre")

df["energy_dance"] = df["energy"] * df["danceability"]

df = pd.concat([df,genre_dummies],axis=1)
feature_cols = [
    "danceability","energy","valence","acousticness",
    "speechiness","instrumentalness","tempo","loudness","year",
    "genre_pop","genre_hip hop","genre_rock","genre_Dance/Electronic",
    "energy_dance"
    ]

X = df[feature_cols]
popular_score = 70
y = np.where(df['popularity'] >= popular_score,1,0)
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.3,
    random_state=42
)

log_model = RandomForestClassifier(
    n_estimators=700,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
log_model.fit(X_train,y_train)

y_pred = log_model.predict(X_test)
print(pd.Series(y_pred).value_counts())

tn,fp,fn,tp = confusion_matrix(y_test,y_pred).ravel()
print("TN:",tn,"TP:",tp,"FN:",fn,"FP:",fp)
acc_normal = (tn+tp)/(tn+fp+fn+tp)
acc_balanced = balanced_accuracy_score(y_test,y_pred)
print("Normal accuracy:",round(acc_normal,4),
      "Balanced accuracy:",round(acc_balanced,4))

y_proba = log_model.predict_proba(X_test)[:,1]
thresholds = np.linspace(0.323,0.325,2000)

best_threshold = 0
best_bal_acc = 0
best_tn = best_fp = best_fn = best_tp = 0

for t in thresholds:
  pred = (y_proba>=t).astype(int)
  tn,fp,fn,tp = confusion_matrix(y_test,pred).ravel()
  bal_acc = balanced_accuracy_score(y_test,pred)

  if bal_acc > best_bal_acc:
    best_bal_acc = bal_acc
    best_threshold = t
    best_tn,best_fp,best_fn,best_tp = tn,fp,fn,tp

print(f"最佳阈值：{best_threshold}")
print(f"最高平衡准确率：{best_bal_acc}")
print(f"普通准确率：{(best_tp+best_tn)/(best_tn+best_fp+best_fn+best_tp)}")
print("TN:",tn,"TP:",tp,"FN:",fn,"FP:",fp)
